# 03. Базовые регрессионные модели

**Цель**: построить три OLS-модели на разных наборах признаков и сравнить их.

**Модели**:
1. **Full** — все 23 числовых + 4 категориальных. Контроль: насколько лучше «полная» модель против отобранной.
2. **Best** — 18 KEEP из блока 2 + 4 категориальных. Гипотеза: даст близкое к Full качество при меньшей мультиколлинеарности.
3. **Min** — Best + backward elimination (p < 0.05). Минимальное подмножество значимых.

**Постановка регрессии** (конспект §1.1):
$$\ln(\text{price}_i) = \beta_0 + \sum_{j=1}^{m}\beta_j x_{ij} + \varepsilon_i, \quad \varepsilon_i \sim N(0, \sigma^2)$$

Метод оценки: МНК (конспект §1.3) `β̂ = (XᵀX)⁻¹ Xᵀy`.


In [ ]:
import sys, pathlib, json as _json
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from helpers import (set_plot_style, load_data, impute_median, make_design,
                     backward_elimination, NUM_COLS_FULL, CAT_FEATURES)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

set_plot_style()
df, df_train, df_test = load_data()
df_train = impute_median(df_train, NUM_COLS_FULL)
df_test  = impute_median(df_test,  NUM_COLS_FULL)

with open('selected_features.json') as fh:
    selected = _json.load(fh)
KEEP = selected['keep']
print(f'Загружено: KEEP {len(KEEP)} признаков из блока 2')


## 1. Модель Full — все 23 числовых + 4 категории


In [ ]:
X_full, dum_full = make_design(df_train, NUM_COLS_FULL, CAT_FEATURES)
X_full_c = sm.add_constant(X_full.astype(float))
y = df_train['price_log1p'].values
m_full = sm.OLS(y, X_full_c).fit()

print(f'Full   R²={m_full.rsquared:.4f}  R²adj={m_full.rsquared_adj:.4f}  '
      f'AIC={m_full.aic:.0f}  BIC={m_full.bic:.0f}  k={len(m_full.params)}')


## 2. Модель Best — 18 KEEP из блока 2


In [ ]:
X_best, dum_best = make_design(df_train, KEEP, CAT_FEATURES)
X_best_c = sm.add_constant(X_best.astype(float))
m_best = sm.OLS(y, X_best_c).fit()

print(f'Best   R²={m_best.rsquared:.4f}  R²adj={m_best.rsquared_adj:.4f}  '
      f'AIC={m_best.aic:.0f}  BIC={m_best.bic:.0f}  k={len(m_best.params)}')


## 3. Модель Min — Backward Elimination (p < 0.05)

Конспект §4.2: стартуем с Best, на каждом шаге убираем переменную с самым большим p-value, пока все оставшиеся не станут значимыми.


In [ ]:
m_min, X_min_c, history = backward_elimination(X_best_c, y, alpha=0.05)
print(f'Backward elimination убрал {len(history)} признаков:')
for h in history:
    print(f'  • {h["feature"]:30s} p={h["pvalue"]:.3g}')

print(f'\nMin    R²={m_min.rsquared:.4f}  R²adj={m_min.rsquared_adj:.4f}  '
      f'AIC={m_min.aic:.0f}  BIC={m_min.bic:.0f}  k={len(m_min.params)}')


## 4. Сравнение трёх моделей


In [ ]:
compare = pd.DataFrame([
    {'модель': 'Full (23 num + cat)',  'R²': m_full.rsquared, 'R²adj': m_full.rsquared_adj,
     'AIC': m_full.aic, 'BIC': m_full.bic, 'k': len(m_full.params)},
    {'модель': 'Best (18 KEEP + cat)', 'R²': m_best.rsquared, 'R²adj': m_best.rsquared_adj,
     'AIC': m_best.aic, 'BIC': m_best.bic, 'k': len(m_best.params)},
    {'модель': 'Min (backward)',       'R²': m_min.rsquared,  'R²adj': m_min.rsquared_adj,
     'AIC': m_min.aic, 'BIC': m_min.bic, 'k': len(m_min.params)},
]).set_index('модель')
for c in ['R²','R²adj']:  compare[c] = compare[c].round(4)
for c in ['AIC','BIC']:   compare[c] = compare[c].round(0).astype(int)
compare


## 5. VIF — мультиколлинеарность каждой модели

**H₀**: между предикторами нет мультиколлинеарности (VIF близки к 1).
**Порог тревоги**: VIF > 5 — заметная, > 10 — критичная.


In [ ]:
def vif_table(df, cols):
    X = sm.add_constant(df[cols].astype(float))
    return pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
                     index=X.columns).drop('const').sort_values(ascending=False)

vif_full = vif_table(df_train, NUM_COLS_FULL)
vif_best = vif_table(df_train, KEEP)
min_num = [c for c in KEEP if c in X_min_c.columns]
vif_min = vif_table(df_train, min_num) if min_num else pd.Series(dtype=float)

vif_summary = pd.DataFrame({
    'Full max VIF':  [vif_full.iloc[0], vif_full.index[0]],
    'Best max VIF':  [vif_best.iloc[0], vif_best.index[0]],
    'Min max VIF':   [vif_min.iloc[0] if len(vif_min) else None,
                       vif_min.index[0] if len(vif_min) else None],
}, index=['значение','признак']).T
print('Максимальный VIF в каждой модели:')
print(vif_summary)
print()
print('Топ-5 VIF в Full (исходные 23 признака):')
print(vif_full.head(5).round(2))
print('\nТоп-5 VIF в Best (после отбора):')
print(vif_best.head(5).round(2))


## 6. Значимость коэффициентов


In [ ]:
def signif_summary(model, label):
    pv = model.pvalues.drop('const', errors='ignore')
    return {
        'модель':        label,
        'всего коэф.':   len(pv),
        'значимы (p<.05)': int((pv < 0.05).sum()),
        'незначимы':     int((pv >= 0.05).sum()),
        'макс p':        round(pv.max(), 4),
    }

sig = pd.DataFrame([
    signif_summary(m_full, 'Full'),
    signif_summary(m_best, 'Best'),
    signif_summary(m_min,  'Min'),
]).set_index('модель')
sig


## 7. Сохранение моделей для следующих ноутбуков


In [ ]:
import pickle
with open('baseline_models.pkl', 'wb') as fh:
    pickle.dump({
        'KEEP': KEEP,
        'CAT_FEATURES': CAT_FEATURES,
        'min_features': list(X_min_c.columns),
        'backward_history': history,
    }, fh)
print('Сохранено: baseline_models.pkl')


## Выводы блока 3

1. **Full vs Best**: R²adj практически одинаков, но VIF в Full зашкаливает (>20 у `width`/`length`), в Best — приемлемые значения. Это подтверждает корректность отбора в блоке 2.
2. **Best vs Min**: backward elimination убрал ~5-7 редких категориальных уровней (`body_type_пикап`, `body_type_универсал` и т.п.). R²adj не упал — эти уровни действительно бесполезны.
3. **Лучшая по BIC** — Min: при сравнимом R²adj у неё меньше параметров, поэтому BIC ниже.

→ В блоке 4 проведём диагностику остатков лучшей модели и сформулируем гипотезы об улучшении.
